## Prompt Template

### 1. What is a PromptTemplate?
A **PromptTemplate** is a reusable string template used for building prompts. Instead of writing raw prompts every time, we declare a prompt template with named placeholders (variables) and then fill these variables during the runtime to produce the final prompt. 

#### 1.1 What are the salient features of a PromptTemplate?

* **Reusable**: Keeps one canonical prompt structure which can be reused with different inputs
* **Consistency**: Enforce the same instruction / context formatting across calls
* **Safety & tooling**: Templates can be validated, partially prefilled, and integrated with tools that manage output parsing

#### 1.2 A Simple PromptTemplate

In [3]:
from langchain import PromptTemplate

template = """You are an expert AI Engineer. Answer concisely.

Question: {question}

"""

prompt_template = PromptTemplate(
    input_variables=["question"],
    template=template
)

prompt = prompt_template.format(question="What is LangChain?")
print(prompt)

You are an expert AI Engineer. Answer concisely.

Question: What is LangChain?




If you refer the above example, **question** is a placeholder for input variables. We supply the value to it while creating the prompt. Even though we define the variable name in curly braces in the prompt template, we would need to explicitly provide the variable name in the input_variables list while creating the prompt template. Langchain uses this to validate the inputs passed during prompt creation. If the input variables are not passed or wrong variables are passed or extra variables are passed during prompt creation, we get a **KeyError**

#### 1.3 PromptTemplate with multiple input variables
We can pass multiple input variables during the template creation. Refer to the example below.

In [4]:
template = """
You are a tutor. 
Explain {topic} to a {level} student in a concise manner.
"""

prompt_template = PromptTemplate(
    input_variables=["topic", "level"],
    template=template
)

prompt = prompt_template.format(topic="Quantum Computing", level="beginner")
print(prompt)


You are a tutor. 
Explain Quantum Computing to a beginner student in a concise manner.



#### 1.4 PromptTemplate with prefilled input variables
We can pass prefilled variables by populating partial_variables input during the creation of PromptTemplate. When we pass prefilled variables, the user only need to enter the input variables. 

In [6]:
template = """
You are a {role}.
Answer the following question: {question}
"""

prompt_template = PromptTemplate(
    input_variables=["question"],
    partial_variables={"role": "data scientist"},
    template=template
)

prompt = prompt_template.format(question="What is overfitting in machine learning?")
print(prompt)


You are a data scientist.
Answer the following question: What is overfitting in machine learning?



##### 1.4.1 We do we need prefilled variables?
* This encourages standardized instructions
* It reduces errors in the downstream code
* Makes prompts easier to audit and version

Think of PromptTemplate as
* A typed function
* With named parameters
* That outputs a string
* And fails if inputs are wrong

### 2. Template formats (f-string vs jinja2)
So far we have used templates where we have passed inputs in curly braces {}. This is the default template format in LangChain. This is called **f-string** format

#### 2.1 f-string format
* Simple variable substitutions
* Similar to python's **str.format** 
* It just replaces the variables with input values. There is no other logic implemented.

##### 2.1.1 When to use f-string format
* The prompt structure is static
* We only need to substitute values
* We want clarity and safety

**f-string format is by default used 80 to 90% of the time**

f-string format functionality is intentionally limited to keep things simple and easy to debug. 
* f-string prompts are Deterministic
* Traces in LangSmith are easy to read
* It avoids hidden branching logic inside the prompts. 

#### 2.2 jinja2 format
We have jinja2 format too available in addition to f-string format. It is very powerful and dangerous if misused. 

**LEARN THIS LATER**

### 3. ChatPromptTemplate

ChatPromptTemplate is a container that aggregates multiple message-level PromptTemplates, each associated with a role. While LangChain allows arbitrary roles syntactically, only a fixed set of roles defined by the underlying LLMs have semantic meaning and should be relied upon. 

A ChatPromptTemplate, usually have the below three role based PromptTemplates

|Class|Role|Purpose|
|-----|----|-------|
|SystemMessagePromptTemplate|system|High-level instructions, behavior|
|HumanMessagePromptTemplate|user|User input|
|AIMessagePromptTemplate|assistant|Prior assistant messages (rare but useful)|

In [1]:
from langchain.prompts import (
    ChatPromptTemplate,
    SystemMessagePromptTemplate,
    HumanMessagePromptTemplate
)

system_prompt_template = SystemMessagePromptTemplate.from_template(
    "You are an expert AI Engineer. Provide me answers based on the questions"
)

human_prompt_template = HumanMessagePromptTemplate.from_template("Explain {topic} in a concise manner.")

chat_prompt_template = ChatPromptTemplate.from_messages([system_prompt_template, human_prompt_template])

chat_prompt = chat_prompt_template.format_messages(topic="Reinforcement Learning")
print(chat_prompt)
# chat_prompt is a list of messages

[SystemMessage(content='You are an expert AI Engineer. Provide me answers based on the questions', additional_kwargs={}, response_metadata={}), HumanMessage(content='Explain Reinforcement Learning in a concise manner.', additional_kwargs={}, response_metadata={})]


#### 3.1 Significance of ChatPromptTemplate

* Stronger Instruction Hierarchy
    * system messages have higher priority
    * User input cannot easily override system rules
    * More reliable behavior

* Better Observability (LangSmith)
    * Each message can be seen separately
    * Easier to debug prompt drifts
    * Clear attribution of failures

* Safer composition
    * Chains can add messages without string concatenation
    * Less risk of malformed prompts

#### 3.2 Shorthand Syntax
Langchain provides a shorthand syntax to create the ChatPromptTemplate which is usually preferred over creating separate variables for SystemMessagePromptTemplate and HumanMessagePromptTemplate

In [2]:
chat_prompt_template =ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("user", "What is the capital of France?")
])

chat_prompt = chat_prompt_template.format_messages()
print(chat_prompt)

[SystemMessage(content='You are a helpful assistant.', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is the capital of France?', additional_kwargs={}, response_metadata={})]


### 4 Prompt drift
Prompt drift is the gradual unintended change in an LLM behavior over time, even if we are using the same prompt. 

In other words:
The model starts giving different answers, tone, structure, or quality without we explicitly changing the prompt in one obvious place. 

#### 4.1 Why prompt drift happens?
There is usually no single reason why prompt drift happens. It emerges from **composition and evaluation**.

* **Prompt composition over time**
    
    In real systems, prompts are rarely static things. They are built from:

    * System instructions
    * User Input
    * Retrieved documents (RAG)
    * Tool outputs
    * Memory / conversation history
    * Conditional logic (Jinja branching)

    If any of these components changes, the effective prompt changes. Even small changes accumulate. 

* **Accidental instruction dilution**
    
    If system roles and user inputs are sandwiched in a single prompt (as done in the traditional PromptTemplate), when user instructions accumulate, there is a risk of earlier system constraints to be overridden or diluted. The model's behavior drifts towards the most recent or dominant instruction patterns. 

    **RAG driven drift**
    
    In RAG systems:
    * Different documents are retrieved
    * Ordering can change
    * Chunk size may vary
    * Irrelevant or conflicting changes may seek in.

    In these cases, the prompt template did not change - **the context did**

* **Prompt mutation via code changes**

    * A new system message added "temporarily"
    * A debug instruction forgotten
    * A fallback message injected on error
    * A tool description modified

    Each is reasonable in isolation. Together, they change behavior. 

* **Model Upgrades**

    Even with an identical prompt:
    * Model version changes
    * Tokenization changes
    * System policy changes
    
    This causes behavioral drift which surfaces as prompt drift. 

Crucially:
* No single commit broke the system
* Rollbacks don't obviously fix it
* Developers argue about whether "prompt actually changed"

#### 4.2 What prompt drift looks like in practice?
Typical symptoms:

* Answers get longer or shorter overtime
* Tone becomes less consistent
* The model starts ignoring instructions it used to follow
* Output format compliance degrades
* Hallucination rate increases
* Tests that used to pass now fail intermittently.

#### 4.3 How LangChain and LangSmith address prompt drift

* Structured prompts (LangChain)

    LangChain reduces drift by:
    * Separating system vs human messages
    * Enforcing input variables
    * Avoiding string concatenation
    * Make prompt structure explicit

* Observability (LangSmith)
    * We see the exact final prompt
    * Message-by-message
    * Across time
    * Across versions

#### 4.4 How engineers prevent prompt drift? (best practices)

* **Treat prompts as versioned artifacts**

    * Store prompts in source control
    * Review them like code
    * Avoid "temporary change

* **Keep system prompts minimal and stable**

    * System messages should be short
    * Declarative not conversational
    * Rarely changed

* **Isolate variability**

    * Put variables in human messages and not in system instructions
    
* **Avoid logic heavy prompts**

    * Dont encode workflows inside prompts
    * Use code for logic and prompts for language

### 5 How prompts are executed: LLMs, Chains, and the full execution flow

In this step we will connect the pieces we have already learnt (PromptTemplate, ChatPromptTemplate, message templates) to 

* How the prompt actually run against an LLM
* How LangChain composes, validates, executes and processes results. 

#### 5.1 High-level flow (what happens from template --> model --> text)

* **Define Templates** (PromptTemplate / ChatPromptTemplate / message templates)
* **Validate / collect variables** LangChain checks input_variables.
* **Fill templates with runtime inputs** Produce final prompt text or message list
* **Send to LLM** via a model abstraction (e.g., ChatOpenAI, OpenAI, Anthropic wrappers)
* **Receive model output** (string or AIMessage)
* **Post-process** schema validation, chaining to next step, storing traces in LangSmith, error handling

#### 5.2 Simple example - simple short prompt with PromptTemplate and LLMChain

In [2]:
##Prerequisite code. This is needed for ChatOpenAI to work.
import os
from dotenv import load_dotenv

load_dotenv()  # take environment variables from .env file

## The below command will read the system environment variable and will set it in python's process environment variable
## This is needed because langchain reads the OPENAI_API_KEY from os.environ and not from os.getenv.
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

## The below will be used for langsmith tracking. More about this later. 
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")

## This line enables V2 tracing in langchain which logs detailed traces of what is happening inside the langchain application.
os.environ["LANGCHAIN_TRACING_V2"] = "true"

## This line sets the project name for langchain tracing (langsmith tracking). More about this later.
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")

In [3]:
from langchain import PromptTemplate, LLMChain
from langchain.chat_models import ChatOpenAI

prompt_template = PromptTemplate(
    template="You are a concise assistant. Answer the question concisely: {question}",
    input_variables=["question"]
)

llm = ChatOpenAI(temperature=0)
chain = LLMChain(llm=llm, prompt=prompt_template)

# Execute the chain
answer = chain.predict(question="What is back propagation?")
print(answer)

C:\Users\ramki\AppData\Local\Temp\ipykernel_18904\2101623042.py:10: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(llm=llm, prompt=prompt_template)


Back propagation is a method used in artificial neural networks to adjust the weights of the network by calculating the gradient of the loss function with respect to the weights.


#### 5.3 LLMChain
**LLMChain** is a runnable unit that binds a prompt template to an LLM and executes them together. 

**LLMChain** connects three things
1. **A Prompt Template**

    * **PromptTemplate** or **ChatPromptTemplate**
    
2. **An LLM Wrapper**

    * **ChatOpenAI**, **OpenAI**, **Anthropic** etc

3. **An execution interface**

    * **.run()**, **.predict()**, **.invoke()**

##### 5.3.1 Why is it called a "Chain"
It is a called a chain because we can chain multiple **LLMChain**s together. Each chain has its own prompt, own LLM configuration and is independently testable. 

##### 5.3.2 LLMChain execution interfaces
LangChain historically exposed three ways to execute a chain. .run(), .predict() and .invoke(). They overlap in capability, but exist for **different abstraction levels and evolution stages** of the framework.

* **.run()** - the simplest, legacy, convenience interface

    **.run()** is the oldest and simplest execution method. 

    * Accepts keyword arguments
    * Returns a single output value
    * Assumes the chain has exactly one output key
    * Designed for quick scripts and demos

    Internally, .run() does

    * input validation
    * calls the chain
    * Extracts a single output

    **Limitations of .run()**

    * breaks when there are multiple outputs
    * does not work well with structured outputs (JSONs, Python dictionaries, typed objects, named fields with guaranteed keys etc)
    * hides input / output schemas
    * does not generalize well to complex pipeline

    **Recommended usage**

    * Learning
    * One-off scripts
    * Simple prototypes
    * Single-input --> Single-output chains

    **When not to use it?**

    * Production systems
    * Multi-step chains
    * Structured outputs
    * Anything you expect to evolve



In [4]:
from langchain import PromptTemplate, LLMChain
from langchain.chat_models import ChatOpenAI

prompt_template = PromptTemplate(
    template="You are a concise assistant. Answer the question concisely: {question}",
    input_variables=["question"]
)

llm = ChatOpenAI(temperature=0)
chain = LLMChain(llm=llm, prompt=prompt_template)

# Execute the chain
answer = chain.run(question="What is back propagation?")
print(answer)

C:\Users\ramki\AppData\Local\Temp\ipykernel_18904\906886352.py:13: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  answer = chain.run(question="What is back propagation?")


Back propagation is a method used in artificial neural networks to adjust the weights of the network by calculating the gradient of the loss function with respect to the weights.


* **.predict()** - Prompt oriented. Slightly more explicit

    **.predict()** is LLMChain-specific and explicitly tied to prompt templates. It makes the mental model clearer. "I am predicting text from an LLM using a prompt". It aligns with how ML engineers think about inference. 

    **Limitations of .predict()**

    * assumes one output
    * hides structure
    * not composable
    * not future proof

    **Recommended Usage**

    * Learning
    * Prototyping

In [5]:
from langchain import PromptTemplate, LLMChain
from langchain.chat_models import ChatOpenAI

prompt_template = PromptTemplate(
    template="You are a concise assistant. Answer the question concisely: {question}",
    input_variables=["question"]
)

llm = ChatOpenAI(temperature=0)
chain = LLMChain(llm=llm, prompt=prompt_template)

# Execute the chain
answer = chain.predict(question="What is back propagation?")
print(answer)

Back propagation is a method used in artificial neural networks to adjust the weights of the network by calculating the gradient of the loss function with respect to the weights.


* **.invoke()** - the modern, standardized execution interface

    .invoke() is part of LangChain's **Runnable** abstraction. This is the future facing API and everything in LangChain is moving toward.

    **Key Characteristics**

    * Accepts a single input object (usually a dict)
    * Returns a structured output
    * Works uniformly across
        * prompts
        * models
        * chains
        * retrievers
        * tools
    * Enables composition, streaming, retries, batching. 

    **Why .invoke() is architecturally superior**

    * inputs are explicit
    * outputs are explicit
    * schemas are inspectable

    **Recommended usage**

    * Production systems
    * testing
    * observability
    * debugging prompt drift

    **Composability**

    We can compose runnables.

    ```python
    chain = prompt | llm | parser
    result = chain.invike("question":"Explain transformers")
    ```

    This is impossible with .run() and .predict()

    **Advanced execution features**

    Only .invoke() supports:

    * batching ( .batch() )
    * streaming ( .stream() )
    * retries
    * fallbacks
    * async execution ( .ainvoke() )

In [7]:
from langchain import PromptTemplate, LLMChain
from langchain.chat_models import ChatOpenAI

prompt_template = PromptTemplate(
    template="You are a concise assistant. Answer the question concisely: {question}",
    input_variables=["question"]
)

llm = ChatOpenAI(temperature=0)
chain = LLMChain(llm=llm, prompt=prompt_template)

# Execute the chain
answer = chain.invoke({"question": "What is back propagation?"})
print(answer)

{'question': 'What is back propagation?', 'text': 'Back propagation is a method used in artificial neural networks to adjust the weights of the network by calculating the gradient of the loss function with respect to the weights.'}


##### 5.3.3 LLMChain Composability

We can connect multiple independent steps into a single pipeline using composability feature of .invoke() method. When we connect these multiple input steps, the output of one step becomes the input of the next, without glue code or manual writing. 

In [10]:
from langchain.prompts import ChatPromptTemplate
from langchain.chat_models import ChatOpenAI

chat_prompt_template =ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("user", "Answer this question: {question}")
])

llm = ChatOpenAI(temperature=0)

##Chain the chat_prompt_template and llm
chain = chat_prompt_template | llm

answer = chain.invoke({"question","What is back propagation"})
print(answer)

content='Backpropagation is a technique used in artificial neural networks to train the model by adjusting the weights of the connections between neurons. It involves calculating the gradient of the loss function with respect to the weights and then updating the weights in the opposite direction of the gradient to minimize the loss. This process is repeated iteratively until the model converges to a solution.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 71, 'prompt_tokens': 30, 'total_tokens': 101, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo', 'system_fingerprint': None, 'finish_reason': 'stop', 'logprobs': None} id='run--feaf303f-b93f-4dcf-a2ef-09ecbce404e5-0'


### 5.4 Out parsing and schema validation
Instead of consuming raw text, production systems parse model outputs into structured data. LangChain provides output parsers for this

In [23]:
from pydantic import BaseModel
from langchain.output_parsers import PydanticOutputParser
from langchain.prompts import ChatPromptTemplate
from langchain.chat_models import ChatOpenAI

##Define the Pydantic model
class Person(BaseModel):
    name: str
    age: int

## Define the chat prompt template
chat_prompt_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    ("user", "Extract a person's name and age from the following text. Return JSON only {text}")
])

## Define the LLM
llm = ChatOpenAI(temperature=0)

## Define the output parser
output_parser = PydanticOutputParser(pydantic_object=Person)

##Chain the chat_prompt_template, llm and output_parser
chain = chat_prompt_template | llm | output_parser

# Execute the chain
answer = chain.invoke({"text": "When I asked john about his age, he said he is born in 1984."})
print(answer)
print(type(answer))

name='John' age=37
<class '__main__.Person'>


In the above example, we have used PydanticOutputParser. This parser, parses the output in a JSON format to a Pydantic model. 

In this example, the method chain.invoke() will validate the output JSON with the model and if it confirms to the model, it returns an object of the model. If it don't confirm, we would get an error. 

#### 5.4.1 Different types of parsers that are available

* Model based parsers

    * PydanticOutputParser

* JSON-based parsers (most common after Pydantic)

    * JsonOutputParser
    * StructuredOutputParser

* Text parsers (can be used for extractions)

    * StrOutputParser
    * RegexParser

* List / delimiter-based parsers

    * CommaSeparatedListOutputParser
    * NumberedListOutputParser

* Enum / choice parsers

    * EnumOutputParser

* Tool / function-based parsers (modern & preferred)

    * OpenAIFunctionsParser
    * JsonKeyOutputFunctionsParser

* Custom parsers (advanced)

    * BaseOutputParser (custom)

        We implement our own parsing logic